# A tiny Python coding agent
ACM University of Birmingham Dubai · Tejesh Vaish · 17 November 2026 GST

**What does checkout() in orders.py do?**

The model choices and answers are **prepared locally**. Python runs two real tools: `list_files()` and `read_file(path)`. Calls and results go back into history until the model answers. No API requests, SDKs or package installs.

Credit: this example uses the model → tool → result loop used at Lumian.

## How an agent uses tools

Five minutes in pairs.

Open `workshop.ipynb` in Colab and connect. Upload **uobd-workshop-student-kit.zip** into the runtime's **Files** pane. Add a temporary code cell before the original cells and run this one-line setup:

```python
from zipfile import ZipFile; ZipFile("uobd-workshop-student-kit.zip").extractall(members=["agent.py", "example_repo/README.md", "example_repo/orders.py"])
```

Open `example_repo/orders.py` in the Files pane. The two tools read the actual `example_repo/` folder beside `agent.py`. For terminal use, follow [Approve and test a patch](README.md#approve-and-test-a-patch).

Before running the cells, read the question with your partner:

1. The normal file list contains `README.md` and `orders.py`. Predict the tool calls, in order.
2. If the list contains `README.md` and `app.py`, where should the loop stop? How many `read_file` calls should run?

Run both prepared cells unchanged with **Shift+Enter**. Inspect each call, the result returned to the model and its final answer. If import fails, check that the setup extracted `agent.py` and both `example_repo/` files into the runtime folder, then rerun.

In [ ]:
from agent import QUESTION, demo

print(QUESTION)
normal_trace, normal_answer = demo()
for event in normal_trace:
    print(event["tool"], event["arguments"], "->", event["result"])
print(normal_answer)

## Prepared missing-file run

The same model and loop now receive the prepared missing-file list `['README.md', 'app.py']`. The example folder stays unchanged. Run this cell and compare its trace with your prediction.

In [ ]:
missing_trace, missing_answer = demo(missing=True)
for event in missing_trace:
    print(event["tool"], event["arguments"], "->", event["result"])
print(missing_answer)

## Inspect the loop

Open `agent.py` and find `ask_repo(question, model, tools)`. The model chooses a named tool, Python executes it, and its result goes into history before the next model turn. An answer stops the loop. Unexpected errors stop execution; the limit is four model turns.

Normal: `list_files()` → `read_file("orders.py")` → an answer explaining that checkout adds the prices of all cart items.

Missing: `list_files()` → `orders.py could not be found.` There are **zero `read_file` calls**.

Discuss: **Why is the file list enough to answer in the missing case?**

## Follow-up: build, break, fix

Same question, same two tools: `list_files()` includes `orders.py`, but `read_file("orders.py")` raises `FileNotFoundError`. What should the model receive?

From the released workshop ZIP root, run `python3 before-fix/test_agent.py` to see the traceback and failing exit, then `python3 test_agent.py` to repeat the identical test and pass. The repo and student ZIP both include `before-fix/`. This follow-up uses `test_agent.py` too; the two cells above use `agent.py` and the `example_repo/` folder extracted by the setup.

The small `try/except` catches `FileNotFoundError` only for `read_file` and sends `{'error': 'file_not_found'}` through the usual tool-result history. The prepared model answers `orders.py could not be read because it is missing.` Other failures still raise; no retry or different path. Reliability starts with reporting the actual tool result, even when it is a failure.

## Review a proposed edit

Optional: propose a change for human review.

Ask to rename `cart` to `items` in `checkout()` in `orders.py`. The same loop lists files and reads the sample, then proposes a diff. Review the parameter and loop source together. Nothing writes or applies the patch.

For **Approve and test a patch**, follow the [terminal instructions in README.md](README.md#approve-and-test-a-patch) after reviewing this preview.

In [ ]:
from agent import PREVIEW_QUESTION, demo

print(PREVIEW_QUESTION)
preview_trace, preview_answer = demo(preview=True)
for event in preview_trace:
    print(event["tool"], event["arguments"], "->", event["result"])
print(preview_answer)